## What this notebook does, and what changed (v3: LLM judge)

**Goal.** Measure whether a pipeline around a small local model (Qwen2.5-Coder-7B) solves more BigCodeBench-Hard tasks than a single model call, and how good an LLM judge (Phi-4 14B) is as the checker inside that pipeline. Final scores always come from the official hidden tests; neither the pipeline nor the judge ever sees them.

**Systems run here**
- **A0 (baseline):** one model call per task.
- **A3 (full pipeline):** builder (writes a spec) -> planner (writes a plan) -> code -> sandbox crash check -> **LLM judge** -> if rejected, **one sampled repair** -> crash check + judge again -> the best candidate is the answer.

**Changes in this version**
1. **The LLM judge replaces the model-written tests.** In the v2 run those tests flagged 25 of A3's 27 correct solutions as failing and accepted wrong code 4 times out of 6. Now `microsoft/phi-4` (a different model family than the generator, loaded in 4-bit) reads the task and the code, and answers `PROBLEMS: ... / VERDICT: PASS|FAIL`. It never sees the spec, the plan or the hidden tests. Its notes go to the repair step. The sandbox only checks that the code loads and the function exists (free and reliable).
2. **One model on the GPU at a time.** Work runs in stages (Qwen for all tasks, then Phi-4 for all tasks, then Qwen for the repairs, then Phi-4 again), so the 14B judge never has to fit next to the generator. Everything is saved after each task, so the run is resumable.
3. **Repairs are sampled, the first try stays greedy.** Repairs use temperature 0.5 with a fixed seed per task and round, so a retry no longer returns the same code (58 of 148 tasks were `stuck` in v2) and reruns still give the same result.
4. **`MAX_TRIES = 2`** (first attempt + 1 repair) to keep the run short.
5. **Kept from v2:** the structured spec (INPUTS, RETURNS, RAISES, BOUNDARIES, REQUIRED LIBRARIES, SIDE EFFECTS, EDGE CASES), the "follow the docstring literally" rule in the code and repair prompts, the full 148-task run, and the install fix.
6. **New analysis cells:** cell 13 splits A3's result into spec+plan effect vs repair effect; cell 14 measures the judge against the hidden tests (precision / recall on the "pass" class); cell 15 (optional) also judges A0's answers.

**Run time (my rough estimate, not measured).** A0 a few minutes per 100 tasks; A3 several hours in total, plus 4 model loads. Rerun cell 9 if the session stops: finished steps are skipped. A *commit* run starts from a clean folder, so it cannot resume. Download `results/` when it finishes.


# BigCodeBench-Hard: baseline vs pipeline + LLM judge - LOCAL Qwen2.5-Coder-7B (generator) and Phi-4 (judge)
One self-contained notebook. No GitHub, no repo, no `pip install -e`.

**What it compares** (same generator, same tasks; only what surrounds the generator changes)
- **A0** one model call
- **A3** + builder (spec) + planner (plan) + crash check + LLM judge + one sampled repair

**Models and settings**
- Generator: `Qwen/Qwen2.5-Coder-7B-Instruct`, fp16, greedy first attempt, sampled repair (temperature 0.5, seeded)
- Judge: `microsoft/phi-4` (14B, 4-bit), greedy, different family than the generator
- Both run locally on the Kaggle GPU (no API, no rate limits). Only one model is loaded at a time.
- Hidden tests are used ONLY for scoring and for grading the judge.

**How to use:** Settings: Accelerator = **GPU T4 x2**, Internet ON. Run cells top to bottom.
After cell 1 (install) do **Run > Restart session** once, then run from cell 2.
With a single 16 GB GPU set `USE_4BIT = True` in cell 2.
Cell 8 is a 2-task smoke test (loads both models, a few minutes): read its output before the full run.
For a quick trial of everything set `SPLIT = "dev"` and `N_TASKS = 5` in cell 2.


In [1]:
# 1. INSTALL (light). Do NOT pip install the full bigcodebench: it pulls vllm.
!pip install -q --no-deps bigcodebench
!pip install -q appdirs datasets fire gradio_client multipledispatch pqdm rich termcolor tree-sitter tree-sitter-python wget tempdir e2b httpx accelerate bitsandbytes
print("Installed. Now: Run > Restart session, then continue from cell 2.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigcodebench 0.2.5 requires anthropic>=0.26.1, which is not installed.
bigcodebench 0.2.5 requires mistralai<1.0.0,>=0.2.0, which is not installed.
bigcodebench 0.2.5 requires vllm, which is not installed.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
Installed. Now: Run > Restart session, then continue from cell 2.


In [2]:
# 2. SETTINGS
import os, json, re, ast, sys, time, random, math, subprocess, tempfile, threading, collections, zlib, gc
from dataclasses import dataclass, field, asdict

MODEL       = "Qwen/Qwen2.5-Coder-7B-Instruct"     # GENERATOR: spec, plan, code and repairs (same model for A0 and A3)
JUDGE_MODEL = "microsoft/phi-4"                    # JUDGE: 14B, a different model family than the generator
USE_4BIT    = False                                # generator fp16 (spans both T4s, as in the first run). Set True on a single 16 GB GPU (P100 / one T4)
JUDGE_4BIT  = True                                 # judge in 4-bit (~9 GB), so it fits on one 16 GB GPU
SYSTEMS_TO_RUN = ["A0", "A1", "A3"]      # A0: direct | A1: direct + judge + repair (no planner/builder) | A3: full pipeline                      # A0 = baseline (1 call) | A3 = spec + plan + code + sandbox crash check + LLM judge + repair
SPLIT       = "all"                                # "all" = every BigCodeBench-Hard task (148) | "dev" (20 tasks) | "test" (128 tasks)
N_TASKS     = None                                 # None = all tasks of the split. Use 5 for a quick trial run
MAX_TRIES   = 2                                    # code attempts per task = first try + 1 repair
REPAIR_TEMPERATURE = 0.5                           # repairs are SAMPLED (seeded) so a retry does not return the same code. First try stays greedy.
REPAIR_TOP_P       = 0.95
JUDGE_A0_OFFLINE   = True                         # cell 15: also judge A0's answers (~148 extra judge calls, ~30+ min) for more judge statistics
OUT_DIR     = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
SUBSET, PROMPT_SPLIT = "hard", "complete"          # BigCodeBench-Hard, docstring prompt
EXECUTION   = "gradio"                             # hidden-test scoring backend hosted on Hugging Face
DEV_SIZE, SPLIT_SEED = 20, 1234

@dataclass
class Config:
    model: str = MODEL
    judge_model: str = JUDGE_MODEL
    temperature: float = 0.0                        # first attempt: greedy decoding (do_sample=False)
    repair_temperature: float = REPAIR_TEMPERATURE  # repair attempts: sampled with a fixed seed per (task, round)
    repair_top_p: float = REPAIR_TOP_P
    max_tokens: int = 768                           # code / repair calls
    max_iters: int = MAX_TRIES - 1                  # number of REPAIRS after the first try
    prompt_version: str = "q7b_v3_phi"              # v3 = Phi-4 judge replaces the generated tests (new result files; old results are not reused)
    stage_max_tokens: dict = field(default_factory=lambda: {"spec": 640, "plan": 512, "judge": 300})
    sandbox: dict = field(default_factory=lambda: {"mem_mb": 8192, "cpu_s": 60, "load_s": 30,
                                                   "test_s": 20, "timeout_s": 120})
cfg = Config()


In [3]:
PROMPTS = {
  # "Follow the docstring literally" rule: shared by the code prompt and the repair prompt
  "system_code": """You are an expert Python programmer. Reply with exactly ONE ```python code block containing all imports, any helper functions from the task, and the complete function with the exact name and signature given in the task. Do NOT write docstrings, comments, example usage or test code. No explanation outside the code block.

FOLLOW THE DOCSTRING LITERALLY:
- Return exactly the type and structure the docstring documents (a tuple is not an array; plain strings are not objects like IPv4Address; use the stated DataFrame columns, order and dtypes).
- Raise exactly the exceptions listed under 'Raises', under exactly the stated conditions, and validate inputs BEFORE using them (check first, then look up).
- Respect boundary conventions: inclusive vs exclusive ends, exact number of items, and never silently skip or drop elements the task asks you to process.
- Use the libraries and functions the task names; do not substitute others.
- Do not add behaviour the docstring does not ask for.""",

  # Structured spec: extracts the details the model tends to miss
  "spec_builder": """You analyze a coding task and write a short checklist spec that another programmer will follow literally. No code. Use exactly these headings, one short line per item, and write 'none' when a heading does not apply:
INPUTS: parameter names, types, defaults.
RETURNS: the exact return type and structure (e.g. a tuple of (A, B); a DataFrame with columns ...; a list of str) and what each element is.
RAISES: every exception type and the exact condition that triggers it, as written in the task.
BOUNDARIES: inclusive or exclusive range ends, how many items are produced, ordering, default values.
REQUIRED LIBRARIES: modules and functions the task names and that must be used.
SIDE EFFECTS: plots, files, random seeds, printing.
EDGE CASES: 3 to 5 cases to handle.
Copy requirements from the task text. Do not invent requirements.""",

  "planner": """You are a senior engineer planning an implementation. Given the task (and an optional spec), write a short numbered plan of 3 to 7 steps: which libraries or functions to use, the order of operations, how to handle edge cases and errors, and what the function returns. If a spec is given, the plan must respect its RETURNS, RAISES and BOUNDARIES lines. One line per step. Plain text only, no code.""",

  "fix": """You are an expert Python programmer fixing code. Reply with exactly ONE ```python code block containing the complete corrected code (imports, helpers, function), no docstrings, no comments, no explanation. Keep following the task text literally: exact return type and structure, exact exceptions and their conditions, exact boundaries, and the libraries the task names.""",

  # v3: the LLM judge (Phi-4) replaces the model-written tests. It sees the task and the code, NOT the spec or the plan.
  "judge": """You are a strict code reviewer. You are given a Python programming task (a function signature with a docstring) and a candidate implementation. Decide whether the candidate would pass the task's hidden unit tests. You cannot run the code, so read it carefully and mentally execute it on the docstring's examples and on edge cases.

Check each of these against the task text:
1. The function name, parameters and defaults match the task exactly.
2. The return type and structure are exactly what the task documents (tuple vs list vs array; plain strings vs objects; the stated DataFrame columns and their order).
3. Every exception listed under 'Raises' is raised under exactly the stated condition, and inputs are validated before they are used.
4. Boundaries are right: inclusive vs exclusive ends, the exact number of items, no element silently skipped or added.
5. The libraries and functions the task names are used, and the imports are present.
6. Required side effects happen (a plot, a file, a random seed) and nothing extra is printed or returned.
7. The logic really implements what the task asks.

Rules: fail the code only for a concrete problem that you can point at in the code, and name it. Do not fail it for style, missing comments, or things the task does not ask for. If you traced the main path and found no concrete problem, pass it.

Reply in exactly this format and nothing else:
PROBLEMS:
- one short line per concrete problem (write 'none' if there is none; at most 3 lines)
VERDICT: PASS
or
VERDICT: FAIL"""
}


In [4]:
# 3. TASKS, TRACE, SANDBOX
import resource  # Linux only
import ast
import re
from dataclasses import dataclass, field


@dataclass
class Task:
    task_id: str
    dataset: str
    prompt: str                      # what the generator sees (never the hidden tests)
    entry_point: str
    trusted_tests: list = field(default_factory=list)   # certainly-correct tests taken from the task text
    extra: dict = field(default_factory=dict)


def extract_code(text: str) -> str:
    """Take the code from a markdown block, even when the closing fence is missing."""
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)```", text, re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:python|py)?[ \t]*\n(.*)", text, re.S)
    if m:
        return m.group(1).strip()
    return text.strip()


def syntax_error(code: str) -> str | None:
    try:
        ast.parse(code)
        return None
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})"
    except Exception as e:  # e.g. null bytes
        return f"{type(e).__name__}: {e}"


import time


class Trace:
    def __init__(self, task_id: str, system: str):
        self.task_id = task_id
        self.system = system
        self.events: list[dict] = []

    def _add(self, etype: str, stage: str, **data) -> str:
        event_id = f"e{len(self.events) + 1:03d}"
        self.events.append({"event_id": event_id, "type": etype, "stage": stage,
                            "ts": round(time.time(), 3), **data})
        return event_id

    def llm(self, stage, system, user, output, tokens, latency, finish_reason) -> str:
        return self._add("llm_call", stage, system=system, user=user, output=output,
                         tokens=tokens, latency=latency, finish_reason=finish_reason)

    def sandbox(self, stage, round_, summary, code) -> str:
        return self._add("sandbox_run", stage, round=round_, code=code, fatal=summary["fatal"],
                         trusted=f'{summary["trusted_pass"]}/{summary["trusted_total"]}',
                         generated=f'{summary["gen_pass"]}/{summary["gen_total"]}',
                         failures=summary["failures"][:6])

    def decision(self, stage, name, **data) -> str:
        return self._add("decision", stage, name=name, **data)

    def tokens(self) -> int:
        return sum(e.get("tokens", 0) for e in self.events if e["type"] == "llm_call")

    def latency(self) -> float:
        return sum(e.get("latency", 0.0) for e in self.events if e["type"] == "llm_call")

    def llm_calls(self) -> int:
        return sum(1 for e in self.events if e["type"] == "llm_call")


import json
import os
import re
import subprocess
import sys
import tempfile

RUNNER = r'''
import json, sys, ast, signal, resource
L = json.load(open("limits.json"))
resource.setrlimit(resource.RLIMIT_AS, (L["mem_mb"] << 20, L["mem_mb"] << 20))
resource.setrlimit(resource.RLIMIT_CPU, (L["cpu_s"], L["cpu_s"]))

def _alarm(signum, frame):
    raise TimeoutError("timed out")
signal.signal(signal.SIGALRM, _alarm)

def out(obj):
    print("@@RESULT@@" + json.dumps(obj))

ns = {"__name__": "solution"}
try:
    signal.alarm(L["load_s"])
    exec(compile(open("solution.py").read(), "solution.py", "exec"), ns)
    signal.alarm(0)
except BaseException as e:
    line = getattr(e, "lineno", None)
    text = e.msg if isinstance(e, SyntaxError) else str(e)
    out({"fatal": type(e).__name__ + ": " + text[:300] + (" (line " + str(line) + ")" if line else ""), "results": []})
    sys.exit(0)

def expected_vs_actual(src):
    try:
        node = ast.parse(src.strip()).body[0]
        t = node.test
        if isinstance(node, ast.Assert) and isinstance(t, ast.Compare) and len(t.ops) == 1 and isinstance(t.ops[0], ast.Eq):
            got = eval(ast.unparse(t.left), ns)
            return " (got " + repr(got)[:100] + ", expected " + ast.unparse(t.comparators[0])[:100] + ")"
    except BaseException:
        pass
    return ""

results = []
for t in json.load(open("tests.json")):
    ok, msg = True, ""
    try:
        signal.alarm(L["test_s"])
        if t["kind"] == "example":
            got = eval(t["src"], ns)
            want_txt = t["want"].strip()
            try:
                want = eval(want_txt, {}); has = True
            except Exception:
                has = False
            ok = (got == want) if has else (repr(got) == want_txt)
            if not ok:
                msg = t["src"].strip() + " -> got " + repr(got)[:100] + ", expected " + want_txt[:100]
        else:
            exec(t["src"], ns)
        signal.alarm(0)
    except AssertionError:
        signal.alarm(0)
        ok, msg = False, "assertion failed: " + t["src"].strip()[:200] + expected_vs_actual(t["src"])
    except BaseException as e:
        signal.alarm(0)
        ok, msg = False, t["src"].strip()[:200] + " raised " + type(e).__name__ + ": " + str(e)[:150]
    results.append({"kind": t["kind"], "ok": ok, "msg": msg})
out({"fatal": None, "results": results})
'''

DEFAULT_LIMITS = {"mem_mb": 1024, "cpu_s": 20, "load_s": 5, "test_s": 3, "timeout_s": 40}


def run_tests(code: str, tests: list, limits: dict | None = None) -> dict:
    lim = {**DEFAULT_LIMITS, **(limits or {})}
    with tempfile.TemporaryDirectory() as d:
        for name, content in (("solution.py", code), ("tests.json", json.dumps(tests)),
                              ("limits.json", json.dumps(lim)), ("runner.py", RUNNER)):
            with open(os.path.join(d, name), "w") as f:
                f.write(content)
        env = {"PATH": os.environ.get("PATH", ""), "HOME": d, "MPLBACKEND": "Agg"}
        try:
            p = subprocess.run([sys.executable, "runner.py"], capture_output=True, text=True,
                               timeout=lim["timeout_s"], cwd=d, env=env)
        except subprocess.TimeoutExpired:
            return {"fatal": "Timeout (infinite loop?)", "results": []}
    m = re.search(r"@@RESULT@@(.*)", p.stdout)
    if not m:
        return {"fatal": "Process crashed: " + p.stderr[-300:], "results": []}
    return json.loads(m.group(1))


def summarize(ev: dict) -> dict:
    """Turn raw sandbox output into counts + flags.

    trusted_ok / all_ok are NEVER true when there are zero trusted tests / zero tests at all
    (the v2 notebook treated "0 of 0 passed" as success, which accepted unverified code).
    """
    res = ev["results"]
    trusted = [r for r in res if r["kind"] == "example"]
    gen = [r for r in res if r["kind"] == "assert"]
    fatal = ev.get("fatal")
    tp, gp = sum(r["ok"] for r in trusted), sum(r["ok"] for r in gen)
    trusted_ok = fatal is None and len(trusted) > 0 and tp == len(trusted)
    all_ok = (fatal is None and len(res) > 0 and tp == len(trusted) and gp == len(gen))
    return {"fatal": fatal, "trusted_total": len(trusted), "trusted_pass": tp,
            "gen_total": len(gen), "gen_pass": gp, "trusted_ok": trusted_ok, "all_ok": all_ok,
            "n_tests": len(res), "failures": [r for r in res if not r["ok"]],
            "score": (0 if fatal else 1, tp, gp)}

In [5]:
# 4. MODELS (generator + judge, ONE loaded at a time) + STAGED PIPELINE
import time, zlib, gc


class LLM:
    """Local Hugging Face model. Greedy decoding by default; seeded sampling only when a temperature is passed."""

    def __init__(self, model_id: str, quant4: bool, label: str, cfg: Config):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        self.cfg, self.torch, self.label, self.model_id = cfg, torch, label, model_id
        self.tok = AutoTokenizer.from_pretrained(model_id)
        kw = dict(device_map="auto")
        if quant4:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16)
        else:
            kw["torch_dtype"] = torch.float16          # T4 does not support bfloat16
        self.model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
        self._lock = threading.Lock()
        print(f"[{label}] {model_id} loaded on {self.model.device} ({'4-bit' if quant4 else 'fp16'})")

    def chat(self, system: str, user: str, trace: Trace, stage: str, temperature=None, seed=None, return_probs=False):
        max_new = self.cfg.stage_max_tokens.get(stage.split("_")[0], self.cfg.max_tokens)
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        gen_kw = dict(max_new_tokens=max_new, pad_token_id=self.tok.eos_token_id)
        if return_probs:
            gen_kw.update(output_scores=True, return_dict_in_generate=True)
        if temperature:                                  # sampled (repairs only)
            gen_kw.update(do_sample=True, temperature=float(temperature), top_p=self.cfg.repair_top_p)
        else:                                            # greedy
            gen_kw.update(do_sample=False)
        t0 = time.time()
        with self._lock, self.torch.no_grad():
            if temperature and seed is not None:
                self.torch.manual_seed(int(seed))
                if self.torch.cuda.is_available():
                    self.torch.cuda.manual_seed_all(int(seed))
            inputs = self.tok(text, return_tensors="pt").to(self.model.device)
            out = self.model.generate(**inputs, **gen_kw)
            
        if return_probs:
            gen_seq = out.sequences[0][inputs.input_ids.shape[1]:]
            reply = self.tok.decode(gen_seq, skip_special_tokens=True)
            p_pass = self._extract_pass_prob(out.scores, gen_seq)
            finish = "length" if len(gen_seq) >= max_new else "stop"
            trace.llm(stage, system, user, reply, int(inputs.input_ids.shape[1] + len(gen_seq)),
                      round(time.time() - t0, 2), finish)
            return reply, p_pass
        else:
            gen = out[0][inputs.input_ids.shape[1]:]
            reply = self.tok.decode(gen, skip_special_tokens=True)
            finish = "length" if len(gen) >= max_new else "stop"
            trace.llm(stage, system, user, reply, int(inputs.input_ids.shape[1] + len(gen)),
                      round(time.time() - t0, 2), finish)
            return reply

    def _extract_pass_prob(self, scores, gen_seq) -> float:
        """Finds the VERDICT token in generated scores and calculates Softmax P(PASS vs FAIL)."""
        if not scores:
            return 0.5
        import torch
        pass_tokens = [self.tok.encode("PASS", add_special_tokens=False)[-1],
                       self.tok.encode(" PASS", add_special_tokens=False)[-1]]
        fail_tokens = [self.tok.encode("FAIL", add_special_tokens=False)[-1],
                       self.tok.encode(" FAIL", add_special_tokens=False)[-1]]
        
        # Look across generated step logits for where PASS or FAIL was emitted
        for step_idx, step_logits in enumerate(scores):
            logits = step_logits[0]
            max_pass = max(logits[tid].item() for tid in set(pass_tokens) if tid < logits.shape[0])
            max_fail = max(logits[tid].item() for tid in set(fail_tokens) if tid < logits.shape[0])
            gen_tok = gen_seq[step_idx].item() if step_idx < len(gen_seq) else None
            if gen_tok in (pass_tokens + fail_tokens) or abs(max_pass - max_fail) > 0.01:
                p = torch.softmax(torch.tensor([max_pass, max_fail], dtype=torch.float32), dim=0)[0].item()
                return round(float(p), 4)
        return 0.5

    def unload(self):
        self.model = None
        gc.collect()
        try:
            self.torch.cuda.empty_cache()
        except Exception:
            pass


class MockLLM:
    """responder(stage, system, user) -> str. For tests and dry runs."""

    def __init__(self, responder):
        self.responder = responder

    def chat(self, system: str, user: str, trace: Trace, stage: str, temperature=None, seed=None, return_probs=False):
        text = self.responder(stage, system, user)
        trace.llm(stage, system, user, text, tokens=len(text) // 4, latency=0.0, finish_reason="stop")
        if return_probs:
            return text, (0.9 if "VERDICT: PASS" in text else 0.1)
        return text


class ModelManager:
    """Keeps exactly one model on the GPU. 'gen' = Qwen generator, 'judge' = Phi-4. Swapping frees the GPU first."""

    def __init__(self, factory=None):
        self.factory = factory or self._default_factory
        self.key, self.llm = None, None

    @staticmethod
    def _default_factory(key):
        model_id, quant4 = (cfg.model, USE_4BIT) if key == "gen" else (cfg.judge_model, JUDGE_4BIT)
        return LLM(model_id, quant4, key, cfg)

    def use(self, key: str):
        if self.key == key:
            return self.llm
        self.free()
        t0 = time.time()
        self.llm, self.key = self.factory(key), key
        print(f"  [model swap] '{key}' ready in {time.time() - t0:.0f} s", flush=True)
        return self.llm

    def free(self):
        if self.llm is not None and hasattr(self.llm, "unload"):
            self.llm.unload()
        self.llm, self.key = None, None
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass


MM = ModelManager()


@dataclass(frozen=True)
class SystemCfg:
    name: str
    builder: bool = False     # spec step added to the generator prompt
    plan: bool = False        # numbered implementation plan added to the generator prompt
    judge: bool = False       # sandbox crash check + LLM judge (replaces the model-written tests of v2)
    repair: bool = False      # repair rejected code (sampled, seeded)
    guard: bool = False       # fix syntax errors before running anything
    fallback: bool = False    # regenerate cleanly (no spec/plan) if the first code can not be parsed


SYSTEMS = {
    "A0": SystemCfg("A0"),
    "A1": SystemCfg("A1", judge=True, repair=True, guard=True, fallback=True),
    "A2": SystemCfg("A2", builder=True, judge=True, repair=True, guard=True, fallback=True),
    "A3": SystemCfg("A3", builder=True, plan=True, judge=True, repair=True, guard=True, fallback=True),
    "A3p": SystemCfg("A3p", plan=True, judge=True, repair=True, guard=True, fallback=True),
}


# ----------------------------------------------------------------------------- helpers
def _gen(llm, system: str, user: str, trace: Trace, stage: str, guard: bool, temperature=None, seed=None) -> str:
    code = extract_code(llm.chat(system, user, trace, stage, temperature=temperature, seed=seed))
    if guard:
        for k in range(2):                               # syntax fixes are always greedy
            err = syntax_error(code)
            if not err:
                break
            fix_user = (f"{user}\n\nYOUR CODE HAD A SYNTAX ERROR: {err}\n\nCODE:\n```python\n{code}\n```\n"
                        "Return the complete corrected code.")
            code = extract_code(llm.chat(system, fix_user, trace, f"{stage}_syntaxfix{k + 1}"))
    return code


def crash_check(code: str, entry_point: str, limits: dict):
    """Free, reliable check: does the code load, and does the function exist? Returns (error_or_None, summary)."""
    if not code.strip():
        return "empty code", {"fatal": "empty code", "trusted_total": 0, "trusted_pass": 0, "gen_total": 0,
                              "gen_pass": 0, "failures": []}
    err = syntax_error(code)
    if err:
        return err, {"fatal": err, "trusted_total": 0, "trusted_pass": 0, "gen_total": 0, "gen_pass": 0, "failures": []}
    s = summarize(run_tests(code, [{"kind": "assert", "src": f"assert callable({entry_point})"}], limits))
    msg = s["fatal"] or (s["failures"][0]["msg"] if s["failures"] else None)
    return msg, s


def judge_user(task: Task, code: str) -> str:
    return (f"TASK:\n{task.prompt}\n\nCODE TO REVIEW:\n```python\n{code}\n```\n\n"
            "AUTOMATIC CHECK: the code loads without errors and the function exists.")


def parse_verdict(text: str):
    hits = re.findall(r"VERDICT\s*:?\s*\**\s*(PASS|FAIL)", text, re.I)
    verdict = hits[-1].upper() if hits else None
    reasons = re.split(r"VERDICT\s*:", text, flags=re.I)[0].strip()
    reasons = re.sub(r"^\s*PROBLEMS\s*:\s*", "", reasons, flags=re.I).strip()
    return verdict, reasons[:700]


def build_feedback(prompt: str, code: str, check: dict) -> str:
    if check.get("crash"):
        problems = "The program failed to load or run: " + str(check["crash"])
    else:
        notes = (check.get("reasons") or "").strip()
        if not notes or notes.lower().startswith("none"):
            notes = ("(no specific problem given) Re-read the task text and check the return type, the exceptions, "
                     "the boundaries and the required libraries one by one.")
        problems = ("A code reviewer rejected the code. The reviewer can be wrong, so verify every point against "
                    "the task text before you change anything:\n" + notes)
    return ("TASK:\n" + prompt + "\n\nYOUR PREVIOUS CODE:\n```python\n" + code + "\n```\n\nPROBLEMS FOUND:\n"
            + problems + "\n\nFix the code. Follow the task text literally. Keep everything that is already correct.")


def _result(task: Task, system: str, cfg: Config, trace: Trace, **fields) -> dict:
    judge_tokens = sum(e.get("tokens", 0) for e in trace.events
                       if e["type"] == "llm_call" and e["stage"].startswith("judge"))
    return {"task_id": task.task_id, "dataset": task.dataset, "system": system, "model": cfg.model,
            "judge_model": cfg.judge_model, "prompt_version": cfg.prompt_version, "tokens": trace.tokens(),
            "judge_tokens": judge_tokens, "latency": round(trace.latency(), 2), "llm_calls": trace.llm_calls(),
            "events": trace.events, **fields}


# ----------------------------------------------------------------------------- staged pipeline
# Per task and per round r the work is split in two stages, so only ONE model is on the GPU at a time:
#   generation stage (Qwen): spec + plan + first code (r = 0)  or  a sampled repair (r > 0), then the crash check
#   judge stage (Phi-4):     the judge reads task + code and answers PASS / FAIL with short notes
# Everything is saved after every task, so a stopped run resumes where it stopped.
def _new_rec(task: Task, name: str) -> dict:
    return {"task_id": task.task_id, "system": name, "spec": "", "plan": "", "codes": [], "checks": [], "events": []}


def _read_state(path: str) -> dict:
    st = {}
    if os.path.isfile(path):
        for line in open(path):
            if line.strip():
                r = json.loads(line)
                st[r["task_id"]] = r                  # last line per task wins
    return st


def _save(path: str, rec: dict):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")


def _needs_gen(rec: dict, sc: SystemCfg, r: int) -> bool:
    if r == 0:
        return len(rec["codes"]) == 0
    if not (sc.repair and sc.judge):
        return False
    return len(rec["codes"]) == r and len(rec["checks"]) == r and not rec["checks"][r - 1]["pass"]


def _needs_judge(rec: dict, sc: SystemCfg, r: int) -> bool:
    return sc.judge and len(rec["codes"]) == r + 1 and len(rec["checks"]) == r


def gen_step(task: Task, rec: dict, sc: SystemCfg, llm, r: int):
    tr = Trace(task.task_id, sc.name)
    tr.events = rec["events"]                         # continue the same trace across stages
    if r == 0:
        if sc.builder:
            rec["spec"] = llm.chat(PROMPTS["spec_builder"], task.prompt, tr, "spec")
        if sc.plan:
            rec["plan"] = llm.chat(PROMPTS["planner"], task.prompt + (f"\n\nSpec:\n{rec['spec']}" if rec["spec"] else ""),
                                   tr, "plan")
        sys_gen = PROMPTS["system_code"]
        if rec["spec"]:
            sys_gen += "\n\nTask spec:\n" + rec["spec"]
        if rec["plan"]:
            sys_gen += "\n\nImplementation plan (follow it):\n" + rec["plan"]
        code = _gen(llm, sys_gen, task.prompt, tr, "generate", sc.guard)               # greedy
        if sc.fallback and syntax_error(code):
            tr.decision("generate", "fallback_clean_generation", reason="code still has a syntax error")
            clean = _gen(llm, PROMPTS["system_code"], task.prompt, tr, "generate_fallback", sc.guard)
            if not syntax_error(clean):
                code = clean
        prev = None
    else:
        prev, pch = rec["codes"][r - 1], rec["checks"][r - 1]
        seed = zlib.crc32(f"{task.task_id}:{r}".encode())                              # same seed on every rerun
        code = _gen(llm, PROMPTS["fix"], build_feedback(task.prompt, prev, pch), tr, f"repair_{r}", sc.guard,
                    temperature=cfg.repair_temperature, seed=seed)
    rec["codes"].append(code)
    if not sc.judge:
        return
    if prev is not None and code.strip() == prev.strip():
        tr.decision(f"repair_{r}", "same_code", why="the repair returned the previous code: reuse its verdict")
        rec["checks"].append({**pch, "judge_run": False, "copied": True})
        return
    msg, s = crash_check(code, task.entry_point, cfg.sandbox)
    tr.sandbox(f"sandbox_{r}", r, s, code)
    if msg:
        rec["checks"].append({"crash": msg, "verdict": "FAIL", "reasons": "", "pass": False, "judge_run": False})
    # no crash: the judge stage adds the check


def judge_step(task: Task, rec: dict, sc: SystemCfg, llm, r: int):
    tr = Trace(task.task_id, sc.name)
    tr.events = rec["events"]
    res = llm.chat(PROMPTS["judge"], judge_user(task, rec["codes"][r]), tr, f"judge_{r}", return_probs=True)
    if isinstance(res, tuple):
        text, p_pass = res
    else:
        text, p_pass = res, (0.8 if "PASS" in str(res) else 0.2)
    verdict, reasons = parse_verdict(text)
    tr.decision(f"judge_{r}", "verdict", verdict=verdict, p_pass=p_pass)
    rec["checks"].append({"crash": None, "verdict": verdict, "reasons": reasons, "pass": verdict == "PASS",
                          "p_pass": p_pass, "judge_run": True})


def _fail(rec: dict, kind: str, r: int, e: Exception):
    msg = f"{type(e).__name__}: {e}"[:300]
    rec.setdefault("errors", []).append({"stage": kind, "round": r, "error": msg})
    if kind == "gen":
        while len(rec["codes"]) < r + 1:
            rec["codes"].append("")
        if SYSTEMS[rec["system"]].judge:
            while len(rec["checks"]) < r + 1:
                rec["checks"].append({"crash": "internal error: " + msg, "verdict": "FAIL", "reasons": "",
                                      "pass": False, "judge_run": False})
    else:
        while len(rec["checks"]) < r + 1:
            rec["checks"].append({"crash": None, "verdict": None, "reasons": "judge error: " + msg,
                                  "pass": False, "judge_run": False, "judge_error": True})


def finalize(task: Task, rec: dict, sc: SystemCfg) -> dict:
    tr = Trace(task.task_id, sc.name)
    tr.events = rec["events"]
    codes, checks = rec["codes"], rec["checks"]
    if not sc.judge:                                    # baseline: one call, no checking
        code = codes[0] if codes else ""
        return _result(task, sc.name, cfg, tr, final_code=code, trajectory=[code], candidates=[code], checks=[],
                       self_pass=None, reason="single_call", verification="none", repairs=0,
                       spec="", plan="", tests=[], history=[])
    best, trajectory, history = None, [], []
    for i, code in enumerate(codes):
        ch = checks[i] if i < len(checks) else {"crash": "not checked", "verdict": None, "pass": False}
        score_t = (0 if ch["crash"] else 1, 1 if ch["pass"] else 0)          # runs > judge pass; ties keep the EARLIER code
        if best is None or score_t > best["score"]:
            best = {"code": code, "score": score_t, "i": i}
        trajectory.append(best["code"])                                       # trajectory[0] = first attempt
        history.append({"round": i, "crash": ch["crash"], "verdict": ch["verdict"], "best_score": list(best["score"])})
    best = best or {"code": "", "score": (0, 0), "i": 0}
    ok = bool(best["score"][1])
    if ok:
        reason = "accepted_first" if best["i"] == 0 else "accepted_after_repair"
    elif all(c.get("crash") for c in checks):
        reason = "no_code_runs"
    else:
        reason = "rejected_by_judge"
    return _result(task, sc.name, cfg, tr, final_code=best["code"], trajectory=trajectory or [""], candidates=codes,
                   checks=checks, self_pass=ok, reason=reason, verification="judge", repairs=max(len(codes) - 1, 0),
                   spec=rec["spec"], plan=rec["plan"], tests=[], history=history,
                   judge_errors=sum(1 for c in checks if c.get("judge_error")))


def run_staged(tasks: list, names: list, state_dir: str, mm=None) -> dict:
    mm = mm or MM
    os.makedirs(state_dir, exist_ok=True)
    paths = {n: f"{state_dir}/{n}__{SPLIT}__{cfg.prompt_version}__state.jsonl" for n in names}
    states = {}
    for n in names:
        old = _read_state(paths[n])
        states[n] = {t.task_id: old.get(t.task_id) or _new_rec(t, n) for t in tasks}

    def stage(label, todo, llm, fn, kind, r):
        t0 = time.time()
        print(f"{label}: {len(todo)} to do", flush=True)
        for k, (n, t) in enumerate(todo, 1):
            rec = states[n][t.task_id]
            try:
                fn(t, rec, SYSTEMS[n], llm, r)
            except RuntimeError:
                raise                                   # GPU out of memory etc.: stop everything
            except Exception as e:                      # one bad task must not kill a multi-hour run
                _fail(rec, kind, r, e)
                print(f"  ! {n} {t.task_id}: {type(e).__name__}: {str(e)[:120]}", flush=True)
            _save(paths[n], rec)
            el = time.time() - t0
            print(f"  {label} [{k}/{len(todo)}] {n} {t.task_id} | elapsed {el / 60:.0f} min | "
                  f"~{el / k * (len(todo) - k) / 60:.0f} min left", flush=True)

    for r in range(MAX_TRIES):
        todo = [(n, t) for n in names for t in tasks if _needs_gen(states[n][t.task_id], SYSTEMS[n], r)]
        if todo:
            stage(f"round {r} generate", todo, mm.use("gen"), gen_step, "gen", r)
        todo = [(n, t) for n in names for t in tasks if _needs_judge(states[n][t.task_id], SYSTEMS[n], r)]
        if todo:
            stage(f"round {r} judge", todo, mm.use("judge"), judge_step, "judge", r)
    mm.free()

    out = {}
    for n in names:
        out[n] = [finalize(t, states[n][t.task_id], SYSTEMS[n]) for t in tasks]
        with open(f"{state_dir}/{n}__{SPLIT}__{cfg.prompt_version}.jsonl", "w") as f:
            for rr in out[n]:
                f.write(json.dumps(rr) + "\n")
    return out


In [6]:
# 5. METRICS + DEV/TEST SPLIT
import collections
import math



def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """95% confidence interval for a proportion."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - a) / d, (c + a) / d)


def sign_test(b: int, c: int) -> float:
    """Exact two-sided sign test (= exact McNemar) on discordant pairs b and c."""
    m = b + c
    if m == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(m, i) for i in range(0, min(b, c) + 1)) / 2 ** m)


def summarize_run(records: list, hidden: dict) -> dict:
    """records: result dicts from pipeline.run_task. hidden: {task_id: bool} from hidden scoring."""
    n = len(records)
    passed = sum(1 for r in records if hidden.get(r["task_id"]))
    lo, hi = wilson(passed, n)
    accepted = [r for r in records if r.get("self_pass")]
    rejected = [r for r in records if r.get("self_pass") is False]
    false_accept = [r for r in accepted if not hidden.get(r["task_id"])]
    false_reject = [r for r in rejected if hidden.get(r["task_id"])]

    by_verif = {}
    for v in sorted({r.get("verification") for r in records if r.get("self_pass") is not None}):
        grp = [r for r in records if r.get("verification") == v]
        acc = [r for r in grp if r["self_pass"]]
        by_verif[v] = {"n": len(grp), "accepted": len(acc),
                       "false_accept": sum(1 for r in acc if not hidden.get(r["task_id"])),
                       "pass_rate": round(sum(1 for r in grp if hidden.get(r["task_id"])) / len(grp), 3)}

    final_syntax = sum(1 for r in records if syntax_error(r["final_code"]))
    tokens = sum(r["tokens"] for r in records)
    return {
        "n_tasks": n, "passed": passed, "pass_rate": round(passed / n, 4) if n else 0.0,
        "ci95": [round(lo, 3), round(hi, 3)],
        "avg_tokens": round(tokens / n, 1) if n else 0, "avg_latency_s": round(sum(r["latency"] for r in records) / n, 2) if n else 0,
        "avg_repairs": round(sum(r.get("repairs", 0) for r in records) / n, 3) if n else 0,
        "tokens_per_solved": round(tokens / passed, 1) if passed else None,
        "final_syntax_error_rate": round(final_syntax / n, 3) if n else 0,
        "stop_reasons": dict(collections.Counter(r.get("reason") for r in records)),
        "false_accept": f"{len(false_accept)}/{len(accepted)}",
        "false_reject": f"{len(false_reject)}/{len(rejected)}",
        "by_verification": by_verif,
    }


def compare_runs(hidden_a: dict, hidden_b: dict) -> dict:
    """Paired comparison B vs A on the tasks both runs have."""
    ids = sorted(set(hidden_a) & set(hidden_b))
    fixed = [t for t in ids if hidden_b[t] and not hidden_a[t]]
    broken = [t for t in ids if hidden_a[t] and not hidden_b[t]]
    return {"n_common": len(ids), "fixed_by_B": len(fixed), "broken_by_B": len(broken),
            "sign_test_p": round(sign_test(len(fixed), len(broken)), 4),
            "fixed_ids": fixed, "broken_ids": broken}


import random


def dev_test_split(task_ids: list, dev_size: int = 20, seed: int = 1234) -> tuple[list, list]:
    ids = sorted(task_ids)
    dev = set(random.Random(seed).sample(ids, min(dev_size, len(ids))))
    return [t for t in ids if t in dev], [t for t in ids if t not in dev]


def select(tasks: list, split: str, dev_size: int, seed: int, limit: int | None = None) -> list:
    dev_ids, test_ids = dev_test_split([t.task_id for t in tasks], dev_size, seed)
    wanted = {"dev": set(dev_ids), "test": set(test_ids)}.get(split, {t.task_id for t in tasks})
    chosen = [t for t in tasks if t.task_id in wanted]
    return chosen[:limit] if limit else chosen

def judge_confusion(rows: list) -> dict:
    """rows: dicts with judge_pass (bool) and hidden_pass (bool). The HIDDEN tests are the ground truth."""
    tp = sum(1 for r in rows if r["judge_pass"] and r["hidden_pass"])
    fp = sum(1 for r in rows if r["judge_pass"] and not r["hidden_pass"])        # false accept: judge passed wrong code
    fn = sum(1 for r in rows if not r["judge_pass"] and r["hidden_pass"])        # false reject: judge failed correct code
    tn = sum(1 for r in rows if not r["judge_pass"] and not r["hidden_pass"])
    n = len(rows)
    return {"judged": n, "hidden_pass_rate": round((tp + fn) / n, 3) if n else None,
            "judge_pass": tp + fp, "TP": tp, "FP(false accept)": fp, "FN(false reject)": fn, "TN": tn,
            "precision_on_pass": round(tp / (tp + fp), 3) if tp + fp else None,
            "recall_on_pass": round(tp / (tp + fn), 3) if tp + fn else None,
            "accuracy": round((tp + tn) / n, 3) if n else None}


In [7]:
# 6. BIGCODEBENCH-HARD: load tasks + hidden scoring (official evaluator)
def load_tasks(subset=SUBSET, split=PROMPT_SPLIT):
    from bigcodebench.data import get_bigcodebench
    key = "complete_prompt" if split == "complete" else "instruct_prompt"
    tasks = [Task(task_id=tid, dataset="bigcodebench", prompt=p[key],
                  entry_point=p.get("entry_point", "task_func"), trusted_tests=[],
                  extra={"libs": p.get("libs")})
             for tid, p in get_bigcodebench(subset=subset).items()]
    return sorted(tasks, key=lambda t: int(t.task_id.split("/")[1]))

def score(solutions: dict, work_dir: str, calibrated=True) -> dict:
    """{task_id: bool} from the HIDDEN tests. Never used to guide the pipeline."""
    from bigcodebench.evaluate import evaluate
    from bigcodebench.data import get_bigcodebench
    os.makedirs(work_dir, exist_ok=True)
    samples = os.path.join(work_dir, "samples.jsonl")
    path = samples.replace(".jsonl", "_eval_results.json")

    def write_samples(items):
        with open(samples, "w") as f:
            for tid, c in items.items():
                f.write(json.dumps({"task_id": tid, "solution": c or "pass"}) + "\n")
        if os.path.isfile(path):
            os.remove(path)          # evaluate() reuses an old results file if it finds one

    # 1) ask the server to grade ONLY our tasks: selective_evaluate wants plain numbers "13,15,17"
    write_samples(solutions)
    try:
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples, execution=EXECUTION, calibrated=calibrated,
                 selective_evaluate=",".join(str(int(t.split("/")[1])) for t in solutions))
    except Exception as e:
        # 2) fallback: send a placeholder answer for every task of the subset (slower, always accepted)
        print("selective_evaluate failed (" + str(e)[:200] + ") -> retrying with all tasks padded")
        all_ids = list(get_bigcodebench(subset=SUBSET))
        write_samples({tid: solutions.get(tid) for tid in all_ids})
        evaluate(split=PROMPT_SPLIT, subset=SUBSET, samples=samples, execution=EXECUTION, calibrated=calibrated)
    if not os.path.isfile(path):
        raise RuntimeError("BigCodeBench produced no results file at " + path)
    data = json.load(open(path))["eval"]
    return {tid: bool(data.get(tid)) and data[tid][0]["status"] == "pass" for tid in solutions}

ALL_TASKS = load_tasks()
TASKS = select(ALL_TASKS, SPLIT, DEV_SIZE, SPLIT_SEED, N_TASKS)
print(len(ALL_TASKS), "tasks in BigCodeBench-Hard;", len(TASKS), "selected (split =", SPLIT + ")")

README.md:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

data/v0.1.0_hf-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  551kB            

data/v0.1.0_hf-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/v0.1.1-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  551kB            

data/v0.1.1-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/v0.1.2-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  552kB            

... (6 outputs left out)

Generating v0.1.3 split:   0%|          | 0/148 [00:00<?, ? examples/s]

Generating v0.1.4 split:   0%|          | 0/148 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

148 tasks in BigCodeBench-Hard; 148 selected (split = all)


In [8]:
# 7. CHECK THE SCORING FIRST (reference solutions must pass ~100%). If not, the setup is wrong, not the pipeline.
from bigcodebench.data import get_bigcodebench
_probs = get_bigcodebench(subset=SUBSET)
_ids = list(_probs)[:5]
_ref = {i: _probs[i]["complete_prompt"] + "\n" + _probs[i]["canonical_solution"] for i in _ids}
_res = score(_ref, OUT_DIR + "/_check")
print(sum(_res.values()), "/", len(_ids), "reference solutions pass", _res)

Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 1.000
pass@1:	1.000
5 / 5 reference solutions pass {'BigCodeBench/13': True, 'BigCodeBench/15': True, 'BigCodeBench/17': True, 'BigCodeBench/19': True, 'BigCodeBench/34': True}


In [9]:
# 8. SMOKE TEST: A3 on 2 tasks through ALL stages (Qwen -> Phi-4 -> Qwen -> Phi-4, so 4 model loads: a few minutes each time)
# Read the output: does the judge answer in the PROBLEMS / VERDICT format? Is the repair different from the first code?
SMOKE = run_staged(TASKS[:2], ["A3"], OUT_DIR + "/smoke")
for r in SMOKE["A3"]:
    print("=" * 70)
    print(r["task_id"], "| stop:", r["reason"], "| repairs:", r["repairs"], "| calls:", r["llm_calls"],
          "| tokens:", r["tokens"], f"(judge {r['judge_tokens']})")
    print("finish reasons:", [e["finish_reason"] for e in r["events"] if e["type"] == "llm_call"])
    for k, c in enumerate(r["checks"]):
        print(f"ROUND {k}: crash={c['crash']} | verdict={c['verdict']} | judged={c['judge_run']}")
        print("   judge notes:", (c["reasons"] or "-")[:300].replace("\n", " | "))
    jc = [e for e in r["events"] if e["type"] == "llm_call" and e["stage"].startswith("judge")]
    if jc:
        print("RAW JUDGE OUTPUT (first call):\n" + jc[0]["output"][:600])
    print("repair differs from first code:", len(r["candidates"]) > 1 and r["candidates"][1].strip() != r["candidates"][0].strip())
# finish_reason "length" on a judge call = the verdict was cut off: raise stage_max_tokens["judge"] in cell 2.
# The smoke test is stored in results/smoke, so it never mixes with the full run.


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (17 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[gen] Qwen/Qwen2.5-Coder-7B-Instruct loaded on cuda:0 (fp16)
  [model swap] 'gen' ready in 32 s
round 1 generate: 2 to do
  round 1 generate [1/2] A3 BigCodeBench/13 | elapsed 0 min | ~0 min left
  round 1 generate [2/2] A3 BigCodeBench/15 | elapsed 1 min | ~0 min left


Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

[judge] microsoft/phi-4 loaded on cuda:0 (4-bit)
  [model swap] 'judge' ready in 110 s
round 1 judge: 2 to do
  round 1 judge [1/2] A3 BigCodeBench/13 | elapsed 0 min | ~0 min left
  round 1 judge [2/2] A3 BigCodeBench/15 | elapsed 0 min | ~0 min left
BigCodeBench/13 | stop: rejected_by_judge | repairs: 1 | calls: 6 | tokens: 6966 (judge 2340)
finish reasons: ['stop', 'stop', 'stop', 'stop', 'stop', 'stop']
ROUND 0: crash=None | verdict=FAIL | judged=True
   judge notes: - The exception handling is incorrect: `ftplib.all_errors` is too broad and will catch all FTP-related errors, including login and directory change errors, which should be handled separately. | - The exception message for `ftplib.all_errors` incorrectly states "Failed to connect to FTP server" instead
ROUND 1: crash=None | verdict=FAIL | judged=True
   judge notes: - The exception handling does not match the task's requirements. The task specifies different error messages for connection, login, and directory change fai

In [10]:
# 9. RUN THE SYSTEMS (resumable: rerun this cell if it stops, finished steps are kept)
# Order: round 0 generate (Qwen: A0 for all tasks, then spec+plan+code for A3) -> round 0 judge (Phi-4)
#        -> round 1 repairs of the rejected tasks (Qwen, sampled) -> round 1 judge (Phi-4) -> final answers
os.makedirs(OUT_DIR, exist_ok=True)
RECORDS = run_staged(TASKS, SYSTEMS_TO_RUN, OUT_DIR)
print({n: len(v) for n, v in RECORDS.items()}, "tasks finished")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[gen] Qwen/Qwen2.5-Coder-7B-Instruct loaded on cuda:0 (fp16)
  [model swap] 'gen' ready in 31 s
round 0 generate: 444 to do
  round 0 generate [1/444] A0 BigCodeBench/13 | elapsed 0 min | ~150 min left
  round 0 generate [2/444] A0 BigCodeBench/15 | elapsed 1 min | ~137 min left
  round 0 generate [3/444] A0 BigCodeBench/17 | elapsed 1 min | ~114 min left
  round 0 generate [4/444] A0 BigCodeBench/19 | elapsed 1 min | ~101 min left
  round 0 generate [5/444] A0 BigCodeBench/34 | elapsed 1 min | ~100 min left
  round 0 generate [6/444] A0 BigCodeBench/37 | elapsed 1 min | ~101 min left
  round 0 generate [7/444] A0 BigCodeBench/82 | elapsed 2 min | ~123 min left
  round 0 generate [8/444] A0 BigCodeBench/89 | elapsed 2 min | ~127 min left
  round 0 generate [9/444] A0 BigCodeBench/92 | elapsed 3 min | ~125 min left
  round 0 generate [10/444] A0 BigCodeBench/93 | elapsed 3 min | ~119 min left
  round 0 generate [11/444] A0 BigCodeBench/99 | elapsed 3 min | ~115 min left
  round 0 genera

<unknown>:11: SyntaxWarning: invalid escape sequence '\s'
<unknown>:11: SyntaxWarning: invalid escape sequence '\s'
<unknown>:11: SyntaxWarning: invalid escape sequence '\s'


  round 0 generate [161/444] A1 BigCodeBench/101 | elapsed 46 min | ~80 min left
  round 0 generate [162/444] A1 BigCodeBench/108 | elapsed 46 min | ~80 min left
  round 0 generate [163/444] A1 BigCodeBench/120 | elapsed 46 min | ~80 min left
  round 0 generate [164/444] A1 BigCodeBench/123 | elapsed 47 min | ~79 min left
  round 0 generate [165/444] A1 BigCodeBench/124 | elapsed 47 min | ~79 min left
  round 0 generate [166/444] A1 BigCodeBench/129 | elapsed 47 min | ~79 min left
  round 0 generate [167/444] A1 BigCodeBench/139 | elapsed 47 min | ~78 min left
  round 0 generate [168/444] A1 BigCodeBench/147 | elapsed 48 min | ~78 min left
  round 0 generate [169/444] A1 BigCodeBench/161 | elapsed 48 min | ~78 min left
  round 0 generate [170/444] A1 BigCodeBench/162 | elapsed 48 min | ~78 min left
  round 0 generate [171/444] A1 BigCodeBench/177 | elapsed 48 min | ~77 min left
  round 0 generate [172/444] A1 BigCodeBench/184 | elapsed 49 min | ~78 min left
  round 0 generate [173/444]

<unknown>:10: SyntaxWarning: invalid escape sequence '\W'
<unknown>:10: SyntaxWarning: invalid escape sequence '\W'
<unknown>:10: SyntaxWarning: invalid escape sequence '\W'


  round 0 generate [237/444] A1 BigCodeBench/655 | elapsed 69 min | ~60 min left
  round 0 generate [238/444] A1 BigCodeBench/657 | elapsed 69 min | ~60 min left
  round 0 generate [239/444] A1 BigCodeBench/678 | elapsed 69 min | ~59 min left
  round 0 generate [240/444] A1 BigCodeBench/720 | elapsed 69 min | ~59 min left
  round 0 generate [241/444] A1 BigCodeBench/723 | elapsed 70 min | ~59 min left
  round 0 generate [242/444] A1 BigCodeBench/752 | elapsed 70 min | ~59 min left
  round 0 generate [243/444] A1 BigCodeBench/760 | elapsed 71 min | ~59 min left
  round 0 generate [244/444] A1 BigCodeBench/763 | elapsed 71 min | ~58 min left
  round 0 generate [245/444] A1 BigCodeBench/765 | elapsed 72 min | ~58 min left
  round 0 generate [246/444] A1 BigCodeBench/771 | elapsed 72 min | ~58 min left
  round 0 generate [247/444] A1 BigCodeBench/777 | elapsed 72 min | ~58 min left
  round 0 generate [248/444] A1 BigCodeBench/785 | elapsed 73 min | ~57 min left
  round 0 generate [249/444]

... (16 outputs left out)

<unknown>:6: SyntaxWarning: invalid escape sequence '\W'
<unknown>:6: SyntaxWarning: invalid escape sequence '\W'


  round 1 generate [218/273] A3 BigCodeBench/657 | elapsed 77 min | ~19 min left
  round 1 generate [219/273] A3 BigCodeBench/678 | elapsed 77 min | ~19 min left
  round 1 generate [220/273] A3 BigCodeBench/720 | elapsed 77 min | ~19 min left
  round 1 generate [221/273] A3 BigCodeBench/723 | elapsed 78 min | ~18 min left
  round 1 generate [222/273] A3 BigCodeBench/752 | elapsed 78 min | ~18 min left
  round 1 generate [223/273] A3 BigCodeBench/760 | elapsed 79 min | ~18 min left
  round 1 generate [224/273] A3 BigCodeBench/763 | elapsed 79 min | ~17 min left
  round 1 generate [225/273] A3 BigCodeBench/765 | elapsed 79 min | ~17 min left
  round 1 generate [226/273] A3 BigCodeBench/771 | elapsed 79 min | ~17 min left
  round 1 generate [227/273] A3 BigCodeBench/777 | elapsed 80 min | ~16 min left
  round 1 generate [228/273] A3 BigCodeBench/785 | elapsed 80 min | ~16 min left
  round 1 generate [229/273] A3 BigCodeBench/800 | elapsed 80 min | ~15 min left
  round 1 generate [230/273]

Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

[judge] microsoft/phi-4 loaded on cuda:0 (4-bit)
  [model swap] 'judge' ready in 109 s
round 1 judge: 248 to do
  round 1 judge [1/248] A1 BigCodeBench/13 | elapsed 0 min | ~60 min left
  round 1 judge [2/248] A1 BigCodeBench/15 | elapsed 0 min | ~54 min left
  round 1 judge [3/248] A1 BigCodeBench/17 | elapsed 1 min | ~53 min left
  round 1 judge [4/248] A1 BigCodeBench/34 | elapsed 1 min | ~52 min left
  round 1 judge [5/248] A1 BigCodeBench/37 | elapsed 1 min | ~50 min left
  round 1 judge [6/248] A1 BigCodeBench/89 | elapsed 1 min | ~50 min left
  round 1 judge [7/248] A1 BigCodeBench/93 | elapsed 1 min | ~48 min left
  round 1 judge [8/248] A1 BigCodeBench/99 | elapsed 2 min | ~47 min left
  round 1 judge [9/248] A1 BigCodeBench/100 | elapsed 2 min | ~47 min left
  round 1 judge [10/248] A1 BigCodeBench/101 | elapsed 2 min | ~47 min left
  round 1 judge [11/248] A1 BigCodeBench/108 | elapsed 2 min | ~46 min left
  round 1 judge [12/248] A1 BigCodeBench/120 | elapsed 2 min | ~47 mi

In [11]:
# 10. SCORE WITH THE HIDDEN TESTS + COMPARE
HIDDEN, SUMMARY = {}, {}
for name, recs in RECORDS.items():
    HIDDEN[name] = score({r["task_id"]: r["final_code"] for r in recs}, f"{OUT_DIR}/score_{name}_{SPLIT}")
    SUMMARY[name] = summarize_run(recs, HIDDEN[name])
    json.dump(HIDDEN[name], open(f"{OUT_DIR}/{name}__{SPLIT}__hidden.json", "w"), indent=1)

import pandas as pd
table = pd.DataFrame({n: {"pass_rate": s["pass_rate"], "95% CI": s["ci95"], "passed": f'{s["passed"]}/{s["n_tasks"]}',
                          "avg_tokens": s["avg_tokens"], "avg_repairs": s["avg_repairs"],
                          "syntax_err_rate": s["final_syntax_error_rate"], "false_accept": s["false_accept"],
                          "false_reject": s["false_reject"], "stop_reasons": s["stop_reasons"]}
                      for n, s in SUMMARY.items()}).T
for n, recs in RECORDS.items():
    table.loc[n, "avg_judge_tokens"] = round(sum(r.get("judge_tokens", 0) for r in recs) / len(recs), 1)
display(table)

base = SYSTEMS_TO_RUN[0]
for other in SYSTEMS_TO_RUN[1:]:
    c = compare_runs(HIDDEN[base], HIDDEN[other])
    print(f"{other} vs {base}: fixed {c['fixed_by_B']}, broken {c['broken_by_B']}, sign-test p = {c['sign_test_p']}")
# With few tasks, differences are mostly noise. Look at p and the CI, not only the pass rate.

Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
Read timeout error. Retrying in 4s...
Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.216
Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/


<unknown>:11: SyntaxWarning: invalid escape sequence '\s'
<unknown>:10: SyntaxWarning: invalid escape sequence '\W'


BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.216
Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/


<unknown>:11: SyntaxWarning: invalid escape sequence '\s'
<unknown>:10: SyntaxWarning: invalid escape sequence '\W'


BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.182


<unknown>:10: SyntaxWarning: invalid escape sequence '\W'
<unknown>:6: SyntaxWarning: invalid escape sequence '\W'


,pass_rate,95% CI,passed,avg_tokens,avg_repairs,syntax_err_rate,false_accept,false_reject,stop_reasons,avg_judge_tokens
A0,0.2162,"[0.158, 0.289]",32/148,767.0,0.0,0.0,0/0,0/0,{'single_call': 148},0.0
A1,0.2162,"[0.158, 0.289]",32/148,3521.2,0.905,0.0,14/17,29/131,"{'rejected_by_judge': 123, 'accepted_first': 1...",1800.1
A3,0.1824,"[0.129, 0.252]",27/148,5823.4,0.939,0.0,9/16,20/132,"{'rejected_by_judge': 125, 'accepted_first': 9...",1854.5


A1 vs A0: fixed 0, broken 0, sign-test p = 1.0
A3 vs A0: fixed 11, broken 16, sign-test p = 0.4421


In [12]:
# 11. LOOK AT ONE TASK IN DETAIL (spec, plan, and what the judge said in each round)
name = "A3"
# first task where the judge rejected the first attempt (change to i = 0 for the very first task)
i = next((k for k, r in enumerate(RECORDS[name]) if r["checks"] and not r["checks"][0]["pass"]), 0)
r = RECORDS[name][i]
print(r["task_id"], "| hidden pass:", HIDDEN[name].get(r["task_id"]), "| stop:", r["reason"], "|", r["verification"])
print("SPEC:\n", r["spec"][:500]); print("PLAN:\n", r["plan"][:500])
for k, c in enumerate(r["checks"]):
    print(f"ROUND {k}: crash={c['crash']} | verdict={c['verdict']} | judged={c['judge_run']}\n   notes: {(c['reasons'] or '-')[:400]}")
print("ROUNDS:", r["history"])


BigCodeBench/13 | hidden pass: False | stop: rejected_by_judge | judge
SPEC:
 **INPUTS:**  
- `ftp_server` (str): The FTP server address. Default is `'ftp.dlptest.com'`.  
- `ftp_user` (str): The FTP server username. Default is `'dlpuser'`.  
- `ftp_password` (str): The FTP server password. Default is `'rNrKYTX9g7z3RgJRmxWuGHbeu'`.  
- `ftp_dir` (str): The directory path on the FTP server from which files need to be downloaded. Default is `'/ftp/test'`.

**RETURNS:**  
- `List[str]`: A list of filenames that were attempted to be downloaded from the FTP server.

**RAISES:*
PLAN:
 1. Use `ftplib.FTP` to establish a connection to the FTP server.
2. Attempt to login to the FTP server using the provided credentials.
3. Change to the specified directory on the FTP server.
4. List all files in the current directory.
5. For each file, construct a command string for `wget` to download the file.
6. Execute the `wget` command using `subprocess.run` to download the file.
7. Collect the names of th

In [13]:
# 12. WHY DID TASKS FAIL? Hidden-test error messages for failed tasks (first MAX_SHOW only)
import glob
NAME, MAX_SHOW = "A3", 15
paths = glob.glob(f"{OUT_DIR}/score_{NAME}_{SPLIT}/*eval_results.json")
if not paths:
    print("No eval results file found. Run cell 10 first.")
else:
    ev = json.load(open(paths[0]))["eval"]
    failed = [(tid, v[0]) for tid, v in ev.items() if v and v[0]["status"] != "pass"]
    print(len(failed), "failed tasks of", len(ev))
    for tid, v in failed[:MAX_SHOW]:
        print(tid, "FAIL"); print(str(v.get("details"))[:400]); print("-" * 60)

121 failed tasks of 148
BigCodeBench/120 FAIL
{'test_reproducibility_with_seed': 'Traceback (most recent call last):\n  File "__test__.py", line 38, in test_reproducibility_with_seed\nAssertionError: Lists differ: [\'202[5056 chars]11-26\', \'2020-06-02\', \'2020-08-22\', \'2020-06-10\', \'2020-02-07\'] != [\'202[5056 chars]11-26\', \'2020-06-02\', \'2020-08-22\', \'2020-06-10\']\n\nFirst list contains 1 additional elements.\nFirst extra elem
------------------------------------------------------------
BigCodeBench/161 FAIL
{'test_log_to_csv_content': 'Traceback (most recent call last):\n  File "__test__.py", line 58, in test_log_to_csv_content\n  File "__test__.py", line 31, in task_func\nValueError: No valid log entries found.\n', 'test_partial_correct_log': 'Traceback (most recent call last):\n  File "__test__.py", line 75, in test_partial_correct_log\n  File "__test__.py", line 31, in task_func\nValueError: No va
------------------------------------------------------------
BigCodeB

In [14]:
# 13. DIAGNOSTIC: is A3's loss caused by the spec/plan or by the repair loop?
a0 = {r["task_id"]: r for r in RECORDS["A0"]}
a3 = {r["task_id"]: r for r in RECORDS["A3"]}
first = {t: r["trajectory"][0] for t, r in a3.items()}            # A3 code BEFORE any repair
HIDDEN["A3_first"] = score(first, f"{OUT_DIR}/score_A3_first_{SPLIT}")   # hidden tests on first attempts only

n = lambda h: sum(h.values())
print(f"A0: {n(HIDDEN['A0'])} | A3 first attempt (spec+plan, no repair): {n(HIDDEN['A3_first'])} | A3 final (after repair): {n(HIDDEN['A3'])}  (out of {len(first)})")

def delta(x, y, label):
    fixed  = [t for t in x if not x[t] and y[t]]
    broken = [t for t in x if x[t] and not y[t]]
    print(f"{label}: fixed {len(fixed)}, broken {len(broken)}")
    return broken

delta(HIDDEN["A0"], HIDDEN["A3_first"], "spec+plan effect   (A0 -> A3 first)")
delta(HIDDEN["A3_first"], HIDDEN["A3"], "repair effect      (A3 first -> A3 final)")

changed = [t for t in a3 if a3[t]["final_code"].strip() != first[t].strip()]
print(len(changed), "tasks had their code changed by repair;", sum(HIDDEN["A3"][t] for t in changed), "of those pass at the end")
print(sum(first[t].strip() == a0[t]["final_code"].strip() for t in a3), "tasks where A3's first attempt is identical to A0's code")


Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 0.993
Failed tasks: ['BigCodeBench/590']
pass@1:	0.182
A0: 32 | A3 first attempt (spec+plan, no repair): 27 | A3 final (after repair): 27  (out of 148)
spec+plan effect   (A0 -> A3 first): fixed 11, broken 16
repair effect      (A3 first -> A3 final): fixed 0, broken 0
8 tasks had their code changed by repair; 1 of those pass at the end
0 tasks where A3's first attempt is identical to A0's code


In [15]:
# 14. HOW GOOD IS THE JUDGE? Every candidate that the judge read is compared with the hidden tests (the ground truth).
# Run cell 13 first (it scores A3's first attempts). This cell scores the repaired candidates too.
a3 = {r["task_id"]: r for r in RECORDS["A3"]}
LABELS = {0: HIDDEN["A3_first"]}                                           # round 0 = first attempts
for k in range(1, MAX_TRIES):
    new = {t: r["candidates"][k] for t, r in a3.items()
           if len(r["checks"]) > k and r["checks"][k].get("judge_run")}   # only repaired code that the judge actually read
    if new:
        HIDDEN[f"A3_cand{k}"] = LABELS[k] = score(new, f"{OUT_DIR}/score_A3_cand{k}_{SPLIT}")

rows = [{"task_id": t, "round": k, "judge_pass": c["pass"], "hidden_pass": LABELS[k][t]}
        for t, r in a3.items() for k, c in enumerate(r["checks"])
        if c.get("judge_run") and k in LABELS and t in LABELS[k]]
print("JUDGE vs HIDDEN TESTS (all rounds):", judge_confusion(rows))
for k in sorted(LABELS):
    print(f"  round {k}:", judge_confusion([x for x in rows if x["round"] == k]))

crashed = [t for t, r in a3.items() if r["checks"] and r["checks"][0]["crash"]]
print(f"\ncrash check: {len(crashed)} first attempts failed to load; of those, {sum(LABELS[0][t] for t in crashed)} pass the hidden tests")
print("judge notes unparsed (no VERDICT line):", sum(1 for r in a3.values() for c in r["checks"] if c.get("judge_run") and c["verdict"] is None))
print("\nREFERENCE, v2 run (model-written tests, from your saved output): accepted 6 -> 4 were wrong; "
      "rejected 142 -> 25 actually passed the hidden tests.")
print("The judge is useful if it passes more of the correct solutions (higher recall) without letting more wrong ones through (precision).")


Loaded as API: https://bigcode-bigcodebench-evaluator.hf.space/
BigCodeBench-Complete (Hard)
Groundtruth pass rate: 1.000
pass@1:	0.134
JUDGE vs HIDDEN TESTS (all rounds): {'judged': 267, 'hidden_pass_rate': 0.165, 'judge_pass': 16, 'TP': 7, 'FP(false accept)': 9, 'FN(false reject)': 37, 'TN': 214, 'precision_on_pass': 0.438, 'recall_on_pass': 0.159, 'accuracy': 0.828}
  round 0: {'judged': 140, 'hidden_pass_rate': 0.193, 'judge_pass': 9, 'TP': 6, 'FP(false accept)': 3, 'FN(false reject)': 21, 'TN': 110, 'precision_on_pass': 0.667, 'recall_on_pass': 0.222, 'accuracy': 0.829}
  round 1: {'judged': 127, 'hidden_pass_rate': 0.134, 'judge_pass': 7, 'TP': 1, 'FP(false accept)': 6, 'FN(false reject)': 16, 'TN': 104, 'precision_on_pass': 0.143, 'recall_on_pass': 0.059, 'accuracy': 0.827}

crash check: 8 first attempts failed to load; of those, 0 pass the hidden tests
judge notes unparsed (no VERDICT line): 0

REFERENCE, v2 run (model-written tests, from your saved output): accepted 6 -> 4 wer

In [16]:
# 15. JUDGE THE BASELINE'S (A0) ANSWERS & EVALUATE JUDGE AS A SELECTOR
# Set JUDGE_A0_OFFLINE = True in cell 2/3.
if JUDGE_A0_OFFLINE:
    llm = MM.use("judge")
    rows0, crashed0 = [], 0
    a0_judgments = {}
    print(f"Judging all {len(TASKS)} answers from A0 using Phi-4...")
    for k, t in enumerate(TASKS, 1):
        code = next(r["final_code"] for r in RECORDS["A0"] if r["task_id"] == t.task_id)
        msg, _ = crash_check(code, t.entry_point, cfg.sandbox)
        if msg:
            crashed0 += 1
            a0_judgments[t.task_id] = {"crash": True, "verdict": "FAIL", "pass": False, "p_pass": 0.0}
            continue
        res = llm.chat(PROMPTS["judge"], judge_user(t, code), Trace(t.task_id, "A0judge"), "judge_0", return_probs=True)
        if isinstance(res, tuple):
            text, p_pass = res
        else:
            text, p_pass = res, 0.5
        verdict, reasons = parse_verdict(text)
        is_pass = (verdict == "PASS")
        a0_judgments[t.task_id] = {"crash": False, "verdict": verdict, "pass": is_pass, "p_pass": p_pass, "reasons": reasons}
        rows0.append({"task_id": t.task_id, "judge_pass": is_pass, "hidden_pass": HIDDEN["A0"][t.task_id], "p_pass": p_pass})
        if k % 25 == 0 or k == len(TASKS):
            print(f"  judged {k}/{len(TASKS)} A0 tasks...", flush=True)
    MM.free()
    
    # Save A0 judgments
    with open(f"{OUT_DIR}/A0_judgments_{SPLIT}.json", "w") as f:
        json.dump(a0_judgments, f, indent=2)
    print(f"\nA0 answers: {crashed0} crashed.", "JUDGE vs HIDDEN (on A0):", judge_confusion(rows0))
    
    # -------------------------------------------------------------------------
    # SELECTOR EXPERIMENT: Choosing between A0 code and A3 first attempt
    # -------------------------------------------------------------------------
    print("\n" + "="*60)
    print("=== SELECTOR EXPERIMENT: A0 vs A3 (First Attempt) ===")
    print("="*60)
    a3_recs = {r["task_id"]: r for r in RECORDS["A3"]}
    
    # Ground truth correctness
    h0 = HIDDEN["A0"]
    h3_first = HIDDEN.get("A3_first", {})
    
    n_a0_correct = sum(h0.values())
    n_a3_first_correct = sum(h3_first.values())
    oracle_correct = sum(bool(h0[t] or h3_first.get(t, False)) for t in h0)
    print(f"Standalone A0 Solved:               {n_a0_correct}/{len(h0)} ({n_a0_correct/len(h0)*100:.1f}%)")
    print(f"Standalone A3 First Attempt Solved: {n_a3_first_correct}/{len(h0)} ({n_a3_first_correct/len(h0)*100:.1f}%)")
    print(f"Oracle Selector (Upper Bound):      {oracle_correct}/{len(h0)} ({oracle_correct/len(h0)*100:.1f}%)")
    print("-" * 60)
    
    # Strategy 1: Hard Verdict Selector (Preference: Pick PASS over FAIL; if tie, default to A0)
    # Strategy 2: Continuous Probability Score Selector (Higher P(PASS))
    hard_selected_pass = 0
    prob_selected_pass = 0
    
    for t_id in h0:
        a0_j = a0_judgments.get(t_id, {"pass": False, "p_pass": 0.0})
        # A3 first attempt check
        a3_checks = a3_recs[t_id]["checks"]
        a3_pass = a3_checks[0].get("pass", False) if a3_checks else False
        a3_prob = a3_checks[0].get("p_pass", 0.5 if a3_pass else 0.1) if a3_checks else 0.0
        
        # 1. Hard Verdict Selector
        if a3_pass and not a0_j["pass"]:
            choice_hard = "A3"
        else:
            choice_hard = "A0"  # default to A0 if both pass, both fail, or only A0 passes
            
        is_hard_correct = h3_first.get(t_id, False) if choice_hard == "A3" else h0[t_id]
        if is_hard_correct:
            hard_selected_pass += 1
            
        # 2. Continuous Probability Score Selector
        # Give a slight 0.02 baseline prior to A0 to break neutral ties
        choice_prob = "A3" if (a3_prob > (a0_j.get("p_pass", 0.0) + 0.02)) else "A0"
        is_prob_correct = h3_first.get(t_id, False) if choice_prob == "A3" else h0[t_id]
        if is_prob_correct:
            prob_selected_pass += 1
            
    print(f"Hard Verdict Selector:              {hard_selected_pass}/{len(h0)} ({hard_selected_pass/len(h0)*100:.1f}%)")
    print(f"Continuous Probability Selector:    {prob_selected_pass}/{len(h0)} ({prob_selected_pass/len(h0)*100:.1f}%)")
    print("=" * 60)
else:
    print("skipped (JUDGE_A0_OFFLINE = False)")


Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

[judge] microsoft/phi-4 loaded on cuda:0 (4-bit)
  [model swap] 'judge' ready in 109 s
Judging all 148 answers from A0 using Phi-4...


<unknown>:11: SyntaxWarning: invalid escape sequence '\s'


  judged 25/148 A0 tasks...
  judged 75/148 A0 tasks...


<unknown>:10: SyntaxWarning: invalid escape sequence '\W'


  judged 100/148 A0 tasks...
  judged 125/148 A0 tasks...
  judged 148/148 A0 tasks...

A0 answers: 8 crashed. JUDGE vs HIDDEN (on A0): {'judged': 140, 'hidden_pass_rate': 0.221, 'judge_pass': 14, 'TP': 3, 'FP(false accept)': 11, 'FN(false reject)': 28, 'TN': 98, 'precision_on_pass': 0.214, 'recall_on_pass': 0.097, 'accuracy': 0.721}

=== SELECTOR EXPERIMENT: A0 vs A3 (First Attempt) ===
Standalone A0 Solved:               32/148 (21.6%)
Standalone A3 First Attempt Solved: 27/148 (18.2%)
Oracle Selector (Upper Bound):      43/148 (29.1%)
------------------------------------------------------------
Hard Verdict Selector:              33/148 (22.3%)
Continuous Probability Selector:    34/148 (23.0%)
